In [ ]:
from google.colab import drive

# Mount your Google Drive
drive.mount('/content/drive')
!pip install biopython

In [ ]:
!apt-get update -qq
!apt-get install -y ncbi-blast+


In [ ]:
import os

PROJECT_DIR = "/content/drive/MyDrive/Subtractive Genomics Approach"
WORK_LINK = "/content/work"
!ln -sfn "{PROJECT_DIR}" {WORK_LINK}

DEG_FASTA_PATH = f"{WORK_LINK}/DEG10.aa"
QUERY_FASTA_NAME = "m_tuberculosis_non_homologous.fasta"
OUTPUT_TSV_NAME = "stage3_deg_hits.tsv"

!makeblastdb -in "{DEG_FASTA_PATH}" -dbtype prot -out "{WORK_LINK}/deg_db"

!blastp -query "{WORK_LINK}/{QUERY_FASTA_NAME}" \
        -db "{WORK_LINK}/deg_db" \
        -out "{WORK_LINK}/{OUTPUT_TSV_NAME}" \
        -evalue 1e-5 \
        -num_threads 2 \
        -outfmt "6 qseqid sseqid pident length mismatch gapopen qstart qend sstart send evalue bitscore"

In [ ]:

import pandas as pd


cols = [
    "Query Sequence ID",
    "Subject Sequence ID",
    "Percentage Identity",
    "Alignment Length",
    "mismatch",
    "gapopen",
    "qstart",
    "qend",
    "sstart",
    "send",
    "evalue",
    "bitscore",
]



In [ ]:
df = pd.read_csv("/content/drive/MyDrive/Subtractive Genomics Approach/stage3_deg_hits.tsv", sep="\t", names=cols)
df

In [ ]:
filtered_df = df[(df["Percentage Identity"] >= 30) & (df["bitscore"] >= 100)]
filtered_df

In [ ]:
filtered_df = df[df["evalue"] <= 1e-4]

In [ ]:
best_hits = filtered_df.sort_values("bitscore", ascending=False)

In [ ]:
best_hits=best_hits.drop_duplicates()

In [ ]:
df["gap_percent"] = (df["gapopen"] / df["Alignment Length"]) * 100
df["mismatch_percent"] = (df["mismatch"] / df["Alignment Length"]) * 100

In [ ]:
filtered_bact="/content/drive/MyDrive/Subtractive Genomics Approach/m_tuberculosis_non_homologous.fasta"

In [ ]:
from Bio import SeqIO
with open(filtered_bact, "r") as handle:
    bact_dict = SeqIO.to_dict(SeqIO.parse(handle, "fasta"))
bact_dict

In [ ]:
bact_series = pd.Series(bact_dict)
bact_series

In [ ]:
clean_series = bact_series.drop(best_hits['Query Sequence ID'])
clean_series

In [ ]:
non_homo_bact = list(clean_series)

In [ ]:
len(non_homo_bact)

In [ ]:
from Bio import SeqIO


SeqIO.write(non_homo_bact, "/content/drive/MyDrive/Subtractive Genomics Approach/m_tuberculosis_EDG_filter.fasta", "fasta")